# CVAT Pose Batch Workflow (Upload -> Pre-label -> Review -> COCO Export)

This notebook automates a CVAT-assisted labeling loop:
1. Upload all images from a local folder into a new CVAT task.
2. Optionally import model predictions as initial annotations (pre-labels).
3. Let annotators review and correct labels in CVAT UI.
4. Detect when all jobs are completed.
5. Trigger and download a COCO export.

Expected manual step: after upload/import, annotators complete the task in CVAT before running the export cells.

Predictions import expects a CVAT-supported annotation format (for pose: COCO Keypoints 1.0).

https://docs.cvat.ai/docs/api_sdk/api/

In [7]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import json
import mimetypes
import time
from typing import Iterable

import cv2
import numpy as np
import requests

In [8]:
CVAT_BASE_URL = "http://localhost:8088"
CVAT_API_TOKEN = "qMWmKIthXdxd4fljYSTLNJIL9MRjK6AN9qnnf7Vm"

PROJECT_ID = 4

REPO_ROOT = Path(r"C:\Users\mhjde\source\repos\CHIMP").resolve()

IMAGES_DIR = (REPO_ROOT / "hpe/notebooks/images/to_label").resolve()
EXPORT_DIR = (REPO_ROOT / "hpe/notebooks/exports/cvat").resolve()
PREDICTIONS_ANNOTATION_PATH = (REPO_ROOT / "hpe/notebooks/annotations/predictions_coco_keypoints.json").resolve()
GENERATED_PREDICTIONS_PATH = (REPO_ROOT / "hpe/notebooks/annotations/generated_predictions_coco_keypoints.json").resolve()
TASK_NAME_PREFIX = "pose_batch"

COCO_FORMAT = "COCO Keypoints 1.0"
IMPORT_FORMAT = COCO_FORMAT

# Optional: generate pre-annotations by calling serving_api for each image
USE_SERVING_API_PREDICTIONS = True
SERVING_API_URL = "http://localhost:5254"
MODEL_NAME = "yolo_pose_demo"
STAGE = "production"
SESSION_ID = ""
MODEL_INPUT_SIZE = 640
DETECTION_CONFIDENCE_THRESHOLD = 0.10
KEYPOINT_CONFIDENCE_THRESHOLD = 0.20

# Poll settings
POLL_SECONDS = 15

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
TASK_NAME = f"{TASK_NAME_PREFIX}_{timestamp}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Task name:", TASK_NAME)
print("Images dir:", IMAGES_DIR)
print("Images dir exists:", IMAGES_DIR.exists())
print("Static predictions file:", PREDICTIONS_ANNOTATION_PATH)
print("Static predictions file exists:", PREDICTIONS_ANNOTATION_PATH.exists())
print("Generated predictions file:", GENERATED_PREDICTIONS_PATH)
print("Use serving_api predictions:", USE_SERVING_API_PREDICTIONS)
print("Serving API URL:", SERVING_API_URL)
print("Model name:", MODEL_NAME)
print("Export dir:", EXPORT_DIR)
print("Token configured:", bool(CVAT_API_TOKEN))
print("Project ID:", PROJECT_ID)

Repo root: C:\Users\mhjde\source\repos\CHIMP
Task name: pose_batch_20260410_134844
Images dir: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\images\to_label
Images dir exists: True
Static predictions file: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\annotations\predictions_coco_keypoints.json
Static predictions file exists: False
Generated predictions file: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\annotations\generated_predictions_coco_keypoints.json
Use serving_api predictions: True
Serving API URL: http://localhost:5254
Model name: yolo_pose_demo
Export dir: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\exports\cvat
Token configured: True
Project ID: 4


In [9]:
@dataclass
class CVATContext:
    base_url: str
    session: requests.Session


COCO_PERSON_KEYPOINT_NAMES = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

COCO_PERSON_SKELETON = [
    [16, 14], [14, 12], [17, 15], [15, 13], [12, 13],
    [6, 12], [7, 13], [6, 7], [6, 8], [7, 9],
    [8, 10], [9, 11], [2, 3], [1, 2], [1, 3],
    [2, 4], [3, 5], [4, 6], [5, 7],
]


def cvat_connect(base_url: str, api_token: str) -> CVATContext:
    s = requests.Session()
    s.headers.update({"Authorization": f"Token {api_token}"})

    me = s.get(f"{base_url}/api/users/self", timeout=30)
    me.raise_for_status()

    return CVATContext(base_url=base_url, session=s)


def cvat_request(ctx: CVATContext, method: str, path: str, **kwargs) -> requests.Response:
    url = f"{ctx.base_url}{path}"
    method_upper = method.upper()
    headers = dict(kwargs.pop("headers", {}))
    return ctx.session.request(method=method_upper, url=url, timeout=120, headers=headers, **kwargs)


def wait_for_request(ctx: CVATContext, rq_id: str, poll_seconds: int = 5) -> dict:
    while True:
        resp = cvat_request(ctx, "GET", f"/api/requests/{rq_id}")
        resp.raise_for_status()
        payload = resp.json()
        status = str(payload.get("status", "")).lower()

        if status == "finished":
            return payload
        if status in {"failed", "canceled"}:
            raise RuntimeError(f"CVAT request {rq_id} ended with status={status}: {json.dumps(payload, indent=2)}")

        time.sleep(poll_seconds)


def _iter_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts]
    files.sort()
    return files


def preprocess_image_for_yolo_pose(image_path: Path, size: int = 640) -> tuple[np.ndarray, np.ndarray]:
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    x = resized.transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    return rgb, x


def infer_yolo_pose(
    image_tensor: np.ndarray,
    serving_api_url: str,
    model_name: str,
    stage: str,
    session_id: str = "",
) -> dict:
    infer_url = f"{serving_api_url}/model/{model_name}/infer"
    params = {"stage": stage}
    if session_id:
        params["id"] = session_id

    payload = {"inputs": image_tensor.tolist()}
    try:
        response = requests.post(infer_url, params=params, json=payload, timeout=180)
    except requests.exceptions.ConnectionError as exc:
        raise RuntimeError(
            f"Could not connect to {infer_url}. Ensure serving_api is running and reachable."
        ) from exc

    response.raise_for_status()
    return response.json()


def xywh_to_xyxy(boxes: np.ndarray) -> np.ndarray:
    xyxy = np.zeros_like(boxes)
    xyxy[:, 0] = boxes[:, 0] - (boxes[:, 2] / 2.0)
    xyxy[:, 1] = boxes[:, 1] - (boxes[:, 3] / 2.0)
    xyxy[:, 2] = boxes[:, 0] + (boxes[:, 2] / 2.0)
    xyxy[:, 3] = boxes[:, 1] + (boxes[:, 3] / 2.0)
    return xyxy


def nms_indices(boxes_xyxy: np.ndarray, scores: np.ndarray, iou_threshold: float = 0.45) -> np.ndarray:
    if len(boxes_xyxy) == 0:
        return np.array([], dtype=np.int32)

    x1 = boxes_xyxy[:, 0]
    y1 = boxes_xyxy[:, 1]
    x2 = boxes_xyxy[:, 2]
    y2 = boxes_xyxy[:, 3]
    areas = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = int(order[0])
        keep.append(i)
        if order.size == 1:
            break

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        union = areas[i] + areas[order[1:]] - inter
        iou = np.where(union > 0.0, inter / union, 0.0)

        remaining = np.where(iou <= iou_threshold)[0]
        order = order[remaining + 1]

    return np.array(keep, dtype=np.int32)


def decode_yolo_pose_output(raw_output, conf_threshold: float = 0.10, iou_threshold: float = 0.45) -> list[dict]:
    out = np.asarray(raw_output, dtype=np.float32)
    if out.ndim != 3 or out.shape[0] < 1:
        return []

    pred = np.transpose(out[0], (1, 0)) if out.shape[1] <= out.shape[2] else out[0]
    if pred.shape[1] < 6:
        return []

    scores = pred[:, 4]
    mask = scores >= conf_threshold
    pred = pred[mask]
    scores = scores[mask]
    if len(pred) == 0:
        return []

    boxes_xywh = pred[:, :4]
    boxes_xyxy = xywh_to_xyxy(boxes_xywh)
    keep = nms_indices(boxes_xyxy, scores, iou_threshold=iou_threshold)

    detections = []
    for i in keep:
        row = pred[int(i)]
        keypoint_values = row[5:]
        keypoints = []
        if len(keypoint_values) >= 3 and len(keypoint_values) % 3 == 0:
            kp = keypoint_values.reshape(-1, 3)
            keypoints = [
                {"x": float(point[0]), "y": float(point[1]), "confidence": float(point[2])}
                for point in kp
            ]

        detections.append(
            {
                "confidence": float(row[4]),
                "bbox_xywh": [float(v) for v in row[:4]],
                "keypoints": keypoints,
            }
        )

    return detections


def extract_raw_output_from_result(result: dict) -> np.ndarray:
    predictions = result.get("predictions", {})
    if not isinstance(predictions, dict) or not predictions:
        raise ValueError("No predictions found in serving response")

    raw_outputs = predictions.get("raw", predictions)
    if not isinstance(raw_outputs, dict) or not raw_outputs:
        raise ValueError(
            "Expected prediction tensors under predictions['raw'] or directly under predictions."
        )

    _, first_output_value = next(iter(raw_outputs.items()))
    return np.asarray(first_output_value, dtype=np.float32)


def coco_keypoints_from_detection(
    keypoints: list[dict],
    scale_x: float,
    scale_y: float,
    kp_conf_threshold: float,
) -> tuple[list[float], int]:
    coco_kps: list[float] = []
    num_keypoints = 0

    for i in range(17):
        if i < len(keypoints):
            kp = keypoints[i]
            x = float(kp.get("x", 0.0)) * scale_x
            y = float(kp.get("y", 0.0)) * scale_y
            conf = float(kp.get("confidence", 0.0))
            v = 2 if conf >= kp_conf_threshold else 1
            if conf >= kp_conf_threshold:
                num_keypoints += 1
            coco_kps.extend([x, y, v])
        else:
            coco_kps.extend([0.0, 0.0, 0])

    return coco_kps, num_keypoints


def build_coco_keypoints_predictions(
    image_paths: list[Path],
    serving_api_url: str,
    model_name: str,
    stage: str,
    session_id: str,
    model_input_size: int,
    det_conf_threshold: float,
    kp_conf_threshold: float,
) -> dict:
    coco_images: list[dict] = []
    coco_annotations: list[dict] = []
    next_image_id = 1
    next_annotation_id = 1

    for image_path in image_paths:
        rgb, image_tensor = preprocess_image_for_yolo_pose(image_path, size=model_input_size)
        image_h, image_w = rgb.shape[:2]
        scale_x = float(image_w) / float(model_input_size)
        scale_y = float(image_h) / float(model_input_size)

        coco_images.append(
            {
                "id": next_image_id,
                "file_name": image_path.name,
                "width": image_w,
                "height": image_h,
            }
        )

        result = infer_yolo_pose(
            image_tensor=image_tensor,
            serving_api_url=serving_api_url,
            model_name=model_name,
            stage=stage,
            session_id=session_id,
        )
        raw_output = extract_raw_output_from_result(result)
        detections = decode_yolo_pose_output(raw_output, conf_threshold=det_conf_threshold)

        for det in detections:
            cx, cy, w, h = det["bbox_xywh"]
            x = (float(cx) - float(w) / 2.0) * scale_x
            y = (float(cy) - float(h) / 2.0) * scale_y
            w_scaled = float(w) * scale_x
            h_scaled = float(h) * scale_y

            coco_kps, num_kps = coco_keypoints_from_detection(
                keypoints=det.get("keypoints", []),
                scale_x=scale_x,
                scale_y=scale_y,
                kp_conf_threshold=kp_conf_threshold,
            )

            coco_annotations.append(
                {
                    "id": next_annotation_id,
                    "image_id": next_image_id,
                    "category_id": 1,
                    "iscrowd": 0,
                    "bbox": [x, y, w_scaled, h_scaled],
                    "area": max(0.0, w_scaled) * max(0.0, h_scaled),
                    "num_keypoints": num_kps,
                    "keypoints": coco_kps,
                    "score": float(det.get("confidence", 0.0)),
                }
            )
            next_annotation_id += 1

        next_image_id += 1

    coco = {
        "info": {"description": "CHIMP generated pre-annotations", "version": "1.0"},
        "licenses": [],
        "images": coco_images,
        "annotations": coco_annotations,
        "categories": [
            {
                "id": 1,
                "name": "person",
                "supercategory": "person",
                "keypoints": COCO_PERSON_KEYPOINT_NAMES,
                "skeleton": COCO_PERSON_SKELETON,
            }
        ],
    }
    return coco


def create_task(ctx: CVATContext, project_id: int, task_name: str) -> int:
    payload = {"name": task_name, "project_id": int(project_id)}
    resp = cvat_request(ctx, "POST", "/api/tasks", json=payload)
    if resp.status_code >= 400:
        try:
            details = json.dumps(resp.json(), indent=2)
        except Exception:
            details = resp.text
        raise RuntimeError(f"Failed to create task (status={resp.status_code}): {details}")
    return int(resp.json()["id"])


def upload_images_to_task(ctx: CVATContext, task_id: int, image_paths: Iterable[Path]) -> str | None:
    image_paths = list(image_paths)

    endpoint = f"/api/tasks/{task_id}/data/"
    base_fields = {"image_quality": 100, "sorting_method": "lexicographical"}

    start_resp = cvat_request(ctx, "POST", endpoint, headers={"Upload-Start": "1"})
    if start_resp.status_code not in (200, 202):
        start_resp.raise_for_status()

    files = []
    opened = []
    try:
        for i, p in enumerate(image_paths):
            f = p.open("rb")
            opened.append(f)
            mime = mimetypes.guess_type(p.name)[0] or "application/octet-stream"
            files.append((f"client_files[{i}]", (p.name, f, mime)))

        multiple_resp = cvat_request(
            ctx,
            "POST",
            endpoint,
            headers={"Upload-Multiple": "1"},
            data={"image_quality": str(base_fields["image_quality"])},
            files=files,
        )
        if multiple_resp.status_code not in (200, 202):
            multiple_resp.raise_for_status()
    finally:
        for f in opened:
            f.close()

    finish_resp = cvat_request(
        ctx,
        "POST",
        endpoint,
        headers={"Upload-Finish": "1"},
        json=base_fields,
    )
    finish_resp.raise_for_status()

    payload = finish_resp.json() if finish_resp.content else {}
    return payload.get("rq_id")


def import_annotations_to_task(
    ctx: CVATContext,
    task_id: int,
    annotation_path: Path,
    format_name: str,
) -> str | None:
    if not annotation_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {annotation_path}")

    with annotation_path.open("rb") as f:
        resp = cvat_request(
            ctx,
            "POST",
            f"/api/tasks/{task_id}/annotations/",
            params={"format": format_name},
            files={"annotation_file": (annotation_path.name, f, "application/json")},
        )

    if resp.status_code not in (200, 201, 202):
        try:
            details = json.dumps(resp.json(), indent=2)
        except Exception:
            details = resp.text
        raise RuntimeError(
            f"Failed to import annotations (status={resp.status_code}): {details}"
        )

    payload = resp.json() if resp.content else {}
    return payload.get("rq_id")


def all_jobs_completed(jobs: list[dict]) -> bool:
    if not jobs:
        return False
    done_states = {"completed", "accepted"}
    return all(str(j.get("state", "")).strip().lower().replace(" ", "_") in done_states for j in jobs)


def wait_for_task_completion(ctx: CVATContext, task_id: int, poll_seconds: int) -> list[dict]:
    while True:
        resp = cvat_request(ctx, "GET", f"/api/jobs?task_id={task_id}&page_size=100")
        resp.raise_for_status()
        jobs = resp.json().get("results", [])

        if all_jobs_completed(jobs):
            return jobs

        time.sleep(poll_seconds)


def trigger_export_and_download(
    ctx: CVATContext,
    task_id: int,
    export_dir: Path,
    format_name: str,
    save_images: bool = False,
) -> Path:
    export_dir.mkdir(parents=True, exist_ok=True)

    start_resp = cvat_request(
        ctx,
        "POST",
        f"/api/tasks/{task_id}/dataset/export",
        params={"format": format_name, "save_images": str(bool(save_images)).lower()},
    )
    start_resp.raise_for_status()

    payload = start_resp.json() if start_resp.content else {}
    rq_id = payload.get("rq_id")
    if rq_id:
        request_payload = wait_for_request(ctx, rq_id=rq_id, poll_seconds=5)
        result_url = request_payload.get("result_url")
        if result_url:
            if result_url.startswith("http"):
                dl_resp = ctx.session.get(result_url, timeout=600)
            else:
                dl_resp = cvat_request(ctx, "GET", result_url)
            dl_resp.raise_for_status()
            out_path = export_dir / f"task_{task_id}_{format_name.replace(' ', '_')}.zip"
            out_path.write_bytes(dl_resp.content)
            return out_path

    dl = cvat_request(
        ctx,
        "GET",
        f"/api/tasks/{task_id}/dataset/export",
        params={"format": format_name, "action": "download"},
    )
    dl.raise_for_status()
    out_path = export_dir / f"task_{task_id}_{format_name.replace(' ', '_')}.zip"
    out_path.write_bytes(dl.content)
    return out_path

In [10]:
ctx = cvat_connect(CVAT_BASE_URL, CVAT_API_TOKEN)
image_paths = _iter_images(IMAGES_DIR)
if not image_paths:
    raise RuntimeError(f"No images found in: {IMAGES_DIR}")

resolved_project_id = int(PROJECT_ID)
task_id = create_task(ctx, resolved_project_id, TASK_NAME)
upload_rq_id = upload_images_to_task(ctx, task_id, image_paths)
if upload_rq_id:
    _ = wait_for_request(ctx, upload_rq_id, poll_seconds=2)

annotation_path_for_import: Path | None = None

if USE_SERVING_API_PREDICTIONS:
    print(f"Generating pre-annotations from serving_api for {len(image_paths)} image(s)...")
    coco_predictions = build_coco_keypoints_predictions(
        image_paths=image_paths,
        serving_api_url=SERVING_API_URL,
        model_name=MODEL_NAME,
        stage=STAGE,
        session_id=SESSION_ID,
        model_input_size=MODEL_INPUT_SIZE,
        det_conf_threshold=DETECTION_CONFIDENCE_THRESHOLD,
        kp_conf_threshold=KEYPOINT_CONFIDENCE_THRESHOLD,
    )
    GENERATED_PREDICTIONS_PATH.write_text(json.dumps(coco_predictions), encoding="utf-8")
    annotation_path_for_import = GENERATED_PREDICTIONS_PATH
    print(f"Generated predictions at: {annotation_path_for_import}")
elif PREDICTIONS_ANNOTATION_PATH.exists():
    annotation_path_for_import = PREDICTIONS_ANNOTATION_PATH
    print(f"Using static predictions file: {annotation_path_for_import}")

if annotation_path_for_import is not None and annotation_path_for_import.exists():
    print(f"Importing pre-annotations from: {annotation_path_for_import}")
    import_rq_id = import_annotations_to_task(
        ctx,
        task_id=task_id,
        annotation_path=annotation_path_for_import,
        format_name=IMPORT_FORMAT,
    )
    if import_rq_id:
        _ = wait_for_request(ctx, import_rq_id, poll_seconds=2)
    print("Pre-annotations import complete.")
else:
    print("No predictions available; continuing without pre-annotations.")

print("Upload complete.")
print("Using project id:", resolved_project_id)
print(f"Open CVAT and review task id={task_id}: {CVAT_BASE_URL}/tasks/{task_id}")
print("When review is done (jobs completed), run the next cell.")

Generating pre-annotations from serving_api for 1 image(s)...
Generated predictions at: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\annotations\generated_predictions_coco_keypoints.json
Importing pre-annotations from: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\annotations\generated_predictions_coco_keypoints.json
Pre-annotations import complete.
Upload complete.
Using project id: 4
Open CVAT and review task id=16: http://localhost:8088/tasks/16
When review is done (jobs completed), run the next cell.


In [11]:
completed_jobs = wait_for_task_completion(
    ctx,
    task_id=task_id,
    poll_seconds=POLL_SECONDS,
)
print(f"Task {task_id} completed with {len(completed_jobs)} job(s).")

Task 16 completed with 1 job(s).


In [12]:
export_zip_path = trigger_export_and_download(
    ctx,
    task_id=task_id,
    export_dir=EXPORT_DIR,
    format_name=COCO_FORMAT,
    save_images=False,
)

print("Export downloaded to:", export_zip_path.resolve())

Export downloaded to: C:\Users\mhjde\source\repos\CHIMP\hpe\notebooks\exports\cvat\task_16_COCO_Keypoints_1.0.zip
